# iPro-MP Official Five-Fold Inference: Kaggle GPU

This notebook evaluates the official iPro-MP E. coli model on the same
SeqTrainer validation and held-out test CSVs used by CNN-v2 and DNABERT2.
It performs inference only: there are no epochs and no iPro-MP retraining.

**Required runtime:** NVIDIA T4 GPU. The five official fold checkpoints are
loaded one at a time to stay within T4 memory.

## Fixed scientific contract

- Same GSE144621 predefined split files and labels.
- Seed `42`.
- Official E. coli species ID `10` model.
- Official five-fold ensemble; positive-class probabilities are averaged.
- Overlapping 6-mer tokenization and model length `300`.
- Validation-only MCC threshold selection.
- Held-out test MCC and AUPRC for final comparison.
- Shared SeqTrainer metrics and artifact schema.

The T4 profile uses physical inference batch size `4` and runs validation and
test by default. Train-split predictions are optional because they are not used
for model selection or final reporting. Omitting train inference reduces runtime
without changing validation or test metrics.

## 1. Verify the Kaggle GPU accelerator


In [ ]:
import subprocess
import torch

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    text=True,
).strip()
print("GPU:", gpu_info)
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. In Kaggle, open Notebook options and select "
        "an NVIDIA GPU accelerator before running the notebook."
    )
if not any(name in gpu_info for name in ("T4", "A100")):
    print("Warning: this notebook was designed for a Kaggle T4/A100 GPU. Continuing because CUDA is available.")


## 2. Define reproducible paths and versions


In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
SEQTRAINER_BRANCH = "issue-3-all-model-baselines"
SEQTRAINER_COMMIT = "659cb28a59b44057607329babf16b196b69b1e6f"

REPO_DIR = Path("/kaggle/working/SeqTrainer")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_DATA_DIR = None
LOCAL_DATA_DIR = REPO_DIR / "data" / "promoter_classification"
OUTPUT_DIR = Path("/kaggle/working/ipromp_external_kaggle_ep_genomic_order")

MINIFORGE_DIR = Path("/kaggle/working/miniforge3")
ENV_DIR = Path("/kaggle/working/envs/seqtrainer-ipromp-kaggle")
ENV_PYTHON = ENV_DIR / "bin" / "python"
ENV_SEQTRAINER = ENV_DIR / "bin" / "seqtrainer"
PIP_CACHE_DIR = Path("/kaggle/working/pip-cache")
CONFIG_RELATIVE_PATH = Path(
    "notebooks/final_training/config/ipromp_external_kaggle.toml"
)

LOCAL_MODEL_ROOT = Path("/kaggle/working/ipromp_models")
DNABERT_DIR = LOCAL_MODEL_ROOT / "DNABERT-6"
IPROMP_MODEL_DIR = LOCAL_MODEL_ROOT / "ipromp_ecoli"
RUN_DIR = REPO_DIR / "outputs" / "benchmarks" / "ipromp_external_kaggle_ep_genomic_order"

print("Pinned SeqTrainer commit:", SEQTRAINER_COMMIT)
print("Kaggle input root:", KAGGLE_INPUT_ROOT)
print("Result directory:", OUTPUT_DIR)


## 3. Create the pinned Python 3.10 environment and check out SeqTrainer


In [ ]:
installer = Path("/kaggle/working/Miniforge3-Linux-x86_64.sh")
if not (MINIFORGE_DIR / "bin" / "conda").exists():
    subprocess.run(
        ["wget", "-q", "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh", "-O", str(installer)],
        check=True,
    )
    subprocess.run(["bash", str(installer), "-b", "-p", str(MINIFORGE_DIR)], check=True)

conda = MINIFORGE_DIR / "bin" / "conda"
if not ENV_PYTHON.exists():
    subprocess.run([str(conda), "create", "-y", "-p", str(ENV_DIR), "python=3.10", "pip"], check=True)

install_env = os.environ.copy()
install_env["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
subprocess.run(
    [str(ENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip<25", "setuptools", "wheel"],
    check=True,
    env=install_env,
)
subprocess.run(
    [
        str(ENV_PYTHON), "-m", "pip", "install",
        "numpy==1.24.4", "pandas==2.0.3", "scikit-learn==1.3.2",
        "torch==2.2.2", "transformers==4.29.2", "huggingface_hub<1.0",
        "remotezip", "requests", "packaging", "tomli",
    ],
    check=True,
    env=install_env,
)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--branch", SEQTRAINER_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
    check=True,
    env=install_env,
)

# Use the pinned commit only when it contains the final Kaggle TOML.
# Otherwise keep the branch tip, which may contain the newly added final-training files.
pinned_config = f"{SEQTRAINER_COMMIT}:{CONFIG_RELATIVE_PATH.as_posix()}"
pinned_check = subprocess.run(
    ["git", "-C", str(REPO_DIR), "cat-file", "-e", pinned_config],
    capture_output=True,
    text=True,
    env=install_env,
)
if pinned_check.returncode == 0:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "--detach", SEQTRAINER_COMMIT],
        check=True,
        env=install_env,
    )
    print("Using pinned commit:", SEQTRAINER_COMMIT)
else:
    print(
        "Pinned commit does not contain the final Kaggle TOML; "
        "using the current issue-3 branch tip instead."
    )

subprocess.run(
    [str(ENV_PYTHON), "-m", "pip", "install", "--no-deps", "-e", str(REPO_DIR)],
    check=True,
    env=install_env,
)

preferred_config = REPO_DIR / CONFIG_RELATIVE_PATH
config_candidates = [
    preferred_config,
    REPO_DIR / "notebooks/colab_benchmarks/config/ipromp_external_t4.toml",
    REPO_DIR / "config-examples/benchmarks/ipromp_external.toml",
]
existing_configs = [path for path in config_candidates if path.exists()]
if not existing_configs:
    raise FileNotFoundError(
        "No iPro-MP TOML was found. Push the final-training config or use a branch "
        "that contains notebooks/colab_benchmarks/config/ipromp_external_t4.toml."
    )
CONFIG_PATH = existing_configs[0]
if CONFIG_PATH != preferred_config:
    print("Using tracked compatibility config:", CONFIG_PATH)
else:
    print("Using final Kaggle config:", CONFIG_PATH)


## 4. Verify the pinned environment


In [ ]:
# Verify pinned iPro-MP environment and GPU.
# Why this cell exists:
# - We run this check inside the pinned iPro-MP virtual environment.
# - SeqTrainer imports rdflib through its SBOL/data modules, so rdflib must exist
#   even if this iPro-MP benchmark is using CSV files.

import json
import subprocess
import textwrap

# Install lightweight SeqTrainer import dependency if missing.
subprocess.run(
    [str(ENV_PYTHON), "-m", "pip", "install", "rdflib"],
    check=True,
)

environment_check = r'''
import json
import torch
import transformers
import seqtrainer

payload = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seqtrainer": seqtrainer.__file__,
}

print(json.dumps(payload, indent=2))

if not payload["cuda_available"]:
    raise SystemExit(
        "CUDA is not available inside the pinned iPro-MP environment. "
        "In Kaggle, go to Runtime > Change runtime type > GPU, then rerun setup cells."
    )

gpu_name = payload["cuda_device"] or ""
allowed_gpus = ["T4", "A100"]

if not any(name in gpu_name for name in allowed_gpus):
    print(
        "WARNING: This notebook was designed for T4/A100, but Kaggle assigned: "
        f"{gpu_name}. Continuing because CUDA is available."
    )
'''

result = subprocess.run(
    [str(ENV_PYTHON), "-c", environment_check],
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)

print("\n===== STDOUT =====")
print(result.stdout)

print("\n===== STDERR =====")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Kaggle iPro-MP environment check failed. Read STDOUT/STDERR above."
    )

## 5. Mount Drive and choose persistent locations


In [ ]:
from pathlib import Path

SPLIT_FILES = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

def contains_all_splits(folder):
    folder = Path(folder)
    return folder.is_dir() and all((folder / name).is_file() for name in SPLIT_FILES.values())

def discover_data_directories():
    if KAGGLE_DATA_DIR is not None:
        candidate = Path(KAGGLE_DATA_DIR)
        if not contains_all_splits(candidate):
            raise FileNotFoundError(f"KAGGLE_DATA_DIR is incomplete: {candidate}")
        return [candidate.resolve()]
    matches = []
    if KAGGLE_INPUT_ROOT.exists():
        for train_path in KAGGLE_INPUT_ROOT.rglob(SPLIT_FILES["train"]):
            folder = train_path.parent
            if contains_all_splits(folder):
                resolved = folder.resolve()
                if resolved not in matches:
                    matches.append(resolved)
    return matches

data_directories = discover_data_directories()
if not data_directories:
    raise FileNotFoundError(
        "No attached Kaggle Dataset contains all three canonical split CSVs. "
        "Attach the AIxBio split dataset or set KAGGLE_DATA_DIR manually."
    )
if len(data_directories) > 1:
    print("Multiple complete Kaggle dataset directories were found:")
    for path in data_directories:
        print("-", path)
    raise RuntimeError("Set KAGGLE_DATA_DIR to the intended dataset directory and rerun this cell.")

SOURCE_DATA_DIR = data_directories[0]
print("Using Kaggle dataset folder:", SOURCE_DATA_DIR)


## 6. Locate, stage, and audit the shared CSV splits


In [ ]:
import hashlib
import json
import pandas as pd
import shutil

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

audit = {}
for split, filename in SPLIT_FILES.items():
    source = SOURCE_DATA_DIR / filename
    target = LOCAL_DATA_DIR / filename
    shutil.copy2(source, target)
    frame = pd.read_csv(target)
    missing_columns = {"sequence", "label"}.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{target} is missing required columns: {sorted(missing_columns)}")
    labels = sorted(frame["label"].dropna().unique().tolist())
    if labels != [0, 1]:
        raise ValueError(f"{split} labels must be [0, 1], found {labels}")
    audit[split] = {
        "path": str(target),
        "rows": int(len(frame)),
        "label_counts": {str(k): int(v) for k, v in frame["label"].value_counts().sort_index().items()},
        "sha256": sha256(target),
    }

print(json.dumps(audit, indent=2))
(OUTPUT_DIR / "input_split_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")


## 7. Download or restore the official model files

The five E. coli fold checkpoints are about 1.8 GB in total. The first run can
therefore take time. When Drive caching is enabled, later sessions copy the
same verified files into fast Kaggle-local storage before inference.

In [ ]:
from pathlib import Path
import shutil
import subprocess

KAGGLE_MODEL_DIR = None
CACHE_MODELS = True

LOCAL_MODEL_ROOT.mkdir(parents=True, exist_ok=True)
IPROMP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
DNABERT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FOLDS = [f"10_fold_{fold}.pth" for fold in range(1, 6)]
DNABERT_FILES = ["config.json", "pytorch_model.bin", "vocab.txt"]

def model_cache_complete(root):
    root = Path(root)
    return (
        all((root / "ipromp_ecoli" / name).is_file() for name in EXPECTED_FOLDS)
        and all((root / "DNABERT-6" / name).is_file() for name in DNABERT_FILES)
    )

if KAGGLE_MODEL_DIR is not None and model_cache_complete(KAGGLE_MODEL_DIR):
    print("Restoring model files from attached Kaggle model dataset")
    shutil.copytree(KAGGLE_MODEL_DIR, LOCAL_MODEL_ROOT, dirs_exist_ok=True)
elif not model_cache_complete(LOCAL_MODEL_ROOT):
    print("Downloading official iPro-MP E. coli fold weights and DNABERT-6 files")
    downloader_candidates = [
        REPO_DIR / "notebooks/benchmark_sg/ipromp_benchmark/download_ecoli_weights.py",
        REPO_DIR / "notebooks/benchmarks_sg/ipromp_benchmark/iprompalpine/download_ecoli_weights.py",
    ]
    downloader = next((path for path in downloader_candidates if path.exists()), None)
    if downloader is None:
        raise FileNotFoundError(
            "The iPro-MP downloader was not found in either the current or pinned "
            "benchmark folder layout."
        )
    subprocess.run(
        [str(ENV_PYTHON), str(downloader), "--output-dir", str(IPROMP_MODEL_DIR)],
        check=True,
    )
    download_dnabert = (
        "from huggingface_hub import snapshot_download; "
        f"snapshot_download(repo_id='zhihan1996/DNA_bert_6', local_dir={str(DNABERT_DIR)!r}, "
        "allow_patterns=['config.json','pytorch_model.bin','special_tokens_map.json','tokenizer_config.json','vocab.txt'])"
    )
    subprocess.run([str(ENV_PYTHON), "-c", download_dnabert], check=True)

if not model_cache_complete(LOCAL_MODEL_ROOT):
    raise FileNotFoundError("The required DNABERT-6 files or all five E. coli fold files are incomplete.")

print("Model files ready in:", LOCAL_MODEL_ROOT)
print("iPro-MP E. coli folds:", IPROMP_MODEL_DIR)
print("DNABERT-6 files:", DNABERT_DIR)


In [ ]:
if not model_cache_complete(LOCAL_MODEL_ROOT):
    raise FileNotFoundError("The required DNABERT-6 files or all five E. coli fold files are incomplete.")
print("Verified five-fold iPro-MP model cache.")


## 8. Assert the T4 iPro-MP scientific contract


In [ ]:
import tomllib

with CONFIG_PATH.open("rb") as handle:
    config = tomllib.load(handle)

assert config["experiment"]["seed"] == 42
assert config["split"]["seed"] == 42
assert config["model"]["params"]["species_id"] == 10
assert config["model"]["params"]["folds"] == 5
assert config["model"]["params"]["kmer_size"] == 6
assert config["model"]["params"]["max_length"] == 300
assert config["model"]["params"]["inference_batch_size"] == 4
assert config["training"]["max_epochs"] == 0
assert config["training"]["learning_rate"] == 0.0
assert config["evaluation"]["threshold_strategy"] == "validation_mcc"
assert config["environment"]["precision"] == "float32"
print("Configuration checks passed: five-fold inference, seed 42, validation-MCC thresholding")


## 9. Prepare FASTA files with stable IDs


In [ ]:
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
subprocess.run(
    [str(ENV_SEQTRAINER), "benchmark", "prepare-ipromp", str(CONFIG_PATH), "--base-dir", str(REPO_DIR), "--output-dir", str(RUN_DIR)],
    check=True,
)
for split in ("validation", "test"):
    path = RUN_DIR / "ipromp_fasta" / f"{split}.fasta"
    if not path.exists():
        raise FileNotFoundError(path)
print("FASTA preparation complete")

## 10. Run sequential five-fold inference with restart support

Validation runs first because its probabilities define the single MCC threshold.
Each completed split is copied to Drive. If Kaggle disconnects, rerun from the
top and this cell will restore completed split predictions instead of repeating
them.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

# iPro-MP is inference-only here.
# False = run only validation + test, faster and enough for benchmark comparison.
# True = also run train predictions, useful only if you want full artifact coverage.
RUN_TRAIN_SPLIT = False

splits_to_run = ["validation", "test"]
if RUN_TRAIN_SPLIT:
    splits_to_run.insert(0, "train")

print("Running iPro-MP inference for splits:", splits_to_run)

In [ ]:
PERSISTENT_PREDICTIONS = OUTPUT_DIR / "external_predictions"
LOCAL_PREDICTIONS = RUN_DIR / "external_predictions"
PERSISTENT_PREDICTIONS.mkdir(parents=True, exist_ok=True)
LOCAL_PREDICTIONS.mkdir(parents=True, exist_ok=True)

splits_to_run = ["validation", "test"]
if RUN_TRAIN_SPLIT:
    splits_to_run.insert(0, "train")

for split in splits_to_run:
    local_csv = LOCAL_PREDICTIONS / f"{split}_predictions.csv"
    persistent_csv = PERSISTENT_PREDICTIONS / local_csv.name
    persistent_metadata = persistent_csv.with_suffix(".metadata.json")
    if persistent_csv.exists():
        print(f"Restoring completed {split} predictions from Drive")
        shutil.copy2(persistent_csv, local_csv)
        if persistent_metadata.exists():
            shutil.copy2(persistent_metadata, local_csv.with_suffix(".metadata.json"))
        continue

    command = [
        str(ENV_PYTHON), "-m", "seqtrainer.adapters.ipromp_inference",
        "--input-fasta", str(RUN_DIR / "ipromp_fasta" / f"{split}.fasta"),
        "--output-csv", str(local_csv),
        "--split", split,
        "--dnabert-dir", str(DNABERT_DIR),
        "--model-dir", str(IPROMP_MODEL_DIR),
        "--species-id", "10",
        "--kmer-size", "6",
        "--max-length", "300",
        "--batch-size", "4",
        "--seed", "42",
        "--device", "cuda",
    ]
    print("Running", split, "inference")
    subprocess.run(command, check=True)
    shutil.copy2(local_csv, persistent_csv)
    metadata = local_csv.with_suffix(".metadata.json")
    if metadata.exists():
        shutil.copy2(metadata, persistent_metadata)
    print("Saved", split, "predictions to Drive")

## 11. Evaluate with the shared validation-only threshold policy


In [ ]:
subprocess.run(
    [str(ENV_SEQTRAINER), "benchmark", "run", str(CONFIG_PATH), "--base-dir", str(REPO_DIR), "--output-dir", str(RUN_DIR), "--strict"],
    check=True,
)
shutil.copytree(RUN_DIR, OUTPUT_DIR, dirs_exist_ok=True)
print("Persistent results:", OUTPUT_DIR)

## 12. Inspect metrics and plots


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, roc_curve
import matplotlib.pyplot as plt

metrics_path = OUTPUT_DIR / "metrics.csv"
predictions_path = OUTPUT_DIR / "predictions.csv"
manifest_path = OUTPUT_DIR / "manifest.json"
for path in (metrics_path, predictions_path, manifest_path):
    if not path.exists():
        raise FileNotFoundError(path)

metrics = pd.read_csv(metrics_path)
display(metrics)
test_metrics = metrics.loc[metrics["split"] == "test"].iloc[0]
print("Held-out test MCC:", round(float(test_metrics["mcc"]), 6))
print("Held-out test AUPRC:", round(float(test_metrics["auprc"]), 6))
print("Validation-selected threshold:", round(float(test_metrics["threshold"]), 6))

predictions = pd.read_csv(predictions_path)
test = predictions.loc[predictions["split"] == "test"]
y_true = test["label"].to_numpy()
y_score = test["probability"].to_numpy()
y_pred = test["prediction"].to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=axes[0], colorbar=False)
fpr, tpr, _ = roc_curve(y_true, y_score)
axes[1].plot(fpr, tpr, label=f"AUROC={float(test_metrics['auroc']):.3f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="grey")
axes[1].legend()
precision, recall, _ = precision_recall_curve(y_true, y_score)
axes[2].plot(recall, precision, label=f"AUPRC={float(test_metrics['auprc']):.3f}")
axes[2].legend()
plt.tight_layout()
plt.show()

## 13. Verify the inference artifact contract


In [ ]:
required = [
    "metrics.csv", "metrics.json", "predictions.csv", "manifest.json",
    "ipromp_id_mapping.csv", "external_predictions/validation_predictions.csv",
    "external_predictions/test_predictions.csv",
]
missing = [name for name in required if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing artifacts: {missing}")
print("Completed T4 iPro-MP artifacts:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), path.stat().st_size, "bytes")

## Interpretation

iPro-MP is an external pretrained model, so epochs and learning rate are not used.
This Kaggle version preserves the official five-fold E. coli ensemble and improves
reproducibility by auditing the canonical split hashes, caching model files locally,
running folds sequentially, and selecting the final threshold on validation MCC only.
AUPRC measures probability ranking and cannot be improved by threshold changes alone.
Compare the held-out test result with the earlier Colab run only when the split hashes
and model revision match.
